[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-kernel-pca.ipynb)

# Kernel PCA

*AIBits Academy · Machine Learning End To End · ⚠ Advanced Topic*

Linear PCA can only ever find straight-line structure. When the real structure is curved, the kernel trick — already met on the SVM and Kernel Methods pages — rescues PCA the same way it rescued classification.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

> **⚠ Why This Page Is Marked "Advanced"**
>
> Kernel PCA sits at the intersection of two ideas taught separately earlier in this course: **PCA** (eigen-decomposition of a covariance matrix) and the **kernel trick** (implicitly working in a much higher-dimensional feature space without ever computing the mapping explicitly, first met on the SVM and Kernel Methods pages). If either of those feels shaky, revisit them first — this page assumes both.

## Where Linear PCA Hits a Wall

Every PCA example so far in this course has involved data where the interesting structure is, at least approximately, a straight line or a flat ellipse — exactly the kind of structure a linear projection can capture. But plenty of real structure is genuinely **curved**. The classic diagnostic dataset for exposing this failure mode — used throughout the ML literature for exactly this purpose — is two concentric rings of points, one class per ring:

In [ ]:
from sklearn.datasets import make_circles
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X, y = make_circles(n_samples=400, factor=0.3, noise=0.07, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

# Baseline: raw features, no transform at all
raw_acc = LogisticRegression().fit(X_train, y_train).score(X_test, y_test)

# Linear PCA, keeping both components (no dimensionality lost)
pca = PCA(n_components=2).fit(X_train)
pca_acc = LogisticRegression().fit(pca.transform(X_train), y_train).score(
    pca.transform(X_test), y_test)

print(f"Raw features:  {raw_acc:.4f}")
print(f"Linear PCA:    {pca_acc:.4f}")

Linear PCA doesn't just underperform here — it does **nothing at all**. Both scores are identical (54.17%, barely above the 50% coin-flip baseline for this balanced two-class problem), because PCA's components are just a rotation of the original axes, and rotating two concentric rings doesn't untangle them. No choice of straight line through this data separates the inner ring from the outer one.

## The Kernel Trick, Applied to PCA

Recall the kernel trick from the SVM and Kernel Methods pages: instead of explicitly mapping each point `x` into a much higher-dimensional (or infinite-dimensional) feature space via some function `φ(x)` and working there, a **kernel function** `k(xᵢ, xⱼ) = φ(xᵢ)·φ(xⱼ)` lets you compute the dot product that would result from that mapping — directly from the original points, without ever constructing `φ(x)` itself.

Ordinary PCA eigen-decomposes the covariance matrix of the data. Kernel PCA instead eigen-decomposes the (centered) **kernel matrix** `K`, where `K`<sub>ij</sub> = `k(xᵢ, xⱼ)`:

$$\begin{gathered}1.\ \text{Build the } n\times n \text{ kernel matrix: } K_{ij} = k(x_i,x_j) \text{ for every pair of training points} \\[6pt] 2.\ \text{Center } K \text{ in the (implicit) feature space: } \tilde{K} = K - \mathbf{1}_n K - K\mathbf{1}_n + \mathbf{1}_n K \mathbf{1}_n \ \ (\mathbf{1}_n = n\times n \text{ matrix of } 1/n) \\[6pt] 3.\ \text{Eigen-decompose } \tilde{K} = U\Lambda U^\mathsf{T}, \text{ keep the top } k \text{ eigenvectors/eigenvalues} \\[6pt] 4.\ \text{Project point } x\text{: the } k\text{-th component is } \sum_i \alpha_{ki}\cdot k(x,x_i), \text{ where } \alpha_{ki} \text{ is the } i\text{-th entry of the } k\text{-th eigenvector, scaled by } 1/\sqrt{\lambda_k}\end{gathered}$$

The most common kernel choice for Kernel PCA is the **RBF (Gaussian) kernel**, `k(xᵢ,xⱼ) = exp(−γ‖xᵢ−xⱼ‖²)`, the same kernel already met on the SVM page. Its implicit feature space is infinite-dimensional, which is precisely why it can represent curved structure that no finite linear projection can.

## Kernel PCA in Practice

Re-running the exact same downstream classifier, this time on the RBF-kernel-PCA-transformed features:

In [ ]:
from sklearn.decomposition import KernelPCA

kpca = KernelPCA(n_components=2, kernel='rbf', gamma=10, random_state=42).fit(X_train)
kpca_acc = LogisticRegression().fit(kpca.transform(X_train), y_train).score(
    kpca.transform(X_test), y_test)

print(f"Kernel PCA (RBF, γ=10):  {kpca_acc:.4f}")

| Feature representation | Test accuracy |
|---|---|
| Raw features | 0.5417 |
| Linear PCA (2 components) | 0.5417 — no improvement |
| Kernel PCA (RBF, γ=10) | **0.9917** |

Same classifier, same number of output dimensions (2), same downstream Logistic Regression — the only thing that changed is *which* 2-dimensional projection was used. The RBF kernel implicitly "unrolls" the two rings into a space where a straight line separates them almost perfectly.

## Visualizing the Unrolling

Left: the original two rings — inner (orange) inside outer (blue), no straight line can separate them. Right: the same 400 points after Kernel PCA — the classes now occupy visually distinct regions.

## Choosing a Kernel and Gamma

The RBF kernel's `γ` (gamma) controls how "local" the implicit similarity measure is — the same parameter, with the same effect, as on the SVM page. Too small a `γ` and Kernel PCA barely differs from linear PCA (the kernel is nearly flat everywhere); too large and every point looks equally dissimilar to every other point, destroying useful structure. A polynomial kernel is the other common choice, better suited when the true structure is polynomial rather than radially symmetric. As with SVMs, gamma is normally chosen via cross-validation on a downstream task, not guessed.

## Computational Cost and When to Reach for It

Linear PCA eigen-decomposes a `p×p` covariance matrix (`p` = number of features) — cheap even for large `n`. Kernel PCA eigen-decomposes an `n×n` kernel matrix (`n` = number of training points) — this scales poorly once `n` reaches the tens of thousands, since both the memory (storing an n×n matrix) and the eigen-decomposition cost (roughly O(n³) for a dense solve) grow with the training set size rather than the feature count. In practice, Kernel PCA is reached for when: (1) a linear model needs to be applied to data with known non-linear structure and a fully non-linear model isn't wanted or available, (2) the dataset is small-to-medium sized (the RBF-kernel-SVM guidance from the SVM page about `n` in the thousands, not millions, applies equally here), or (3) as a denoising step — reconstructing points from only their top kernel principal components filters out noise that lies along the lower-variance directions in the implicit feature space.

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Linear PCA cannot unroll rings

Project the two rings onto 2 principal components with ordinary `PCA`, train a `LogisticRegression` on the result and store its test accuracy in `acc_pca`. It stays near chance.

In [ ]:
from sklearn.datasets import make_circles
from sklearn.decomposition import PCA, KernelPCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
Xr, yr = make_circles(n_samples=400, factor=0.3, noise=0.07, random_state=1)
Xa, Xb, ya, yb = train_test_split(Xr, yr, random_state=1)
acc_pca = None   # TODO


In [ ]:
try:
    check("linear PCA is near chance", acc_pca < 0.65)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from sklearn.datasets import make_circles
from sklearn.decomposition import PCA, KernelPCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
Xr, yr = make_circles(n_samples=400, factor=0.3, noise=0.07, random_state=1)
Xa, Xb, ya, yb = train_test_split(Xr, yr, random_state=1)
p = PCA(n_components=2).fit(Xa)
acc_pca = LogisticRegression().fit(p.transform(Xa), ya).score(p.transform(Xb), yb)

```

</details>

### Exercise 2 · Medium · Kernel PCA separates them

Do the same with `KernelPCA(n_components=2, kernel="rbf", gamma=10)` and store the accuracy in `acc_kpca`.

In [ ]:
acc_kpca = None   # TODO


In [ ]:
try:
    check("kernel PCA succeeds", acc_kpca > 0.95)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
k = KernelPCA(n_components=2, kernel="rbf", gamma=10).fit(Xa)
acc_kpca = LogisticRegression().fit(k.transform(Xa), ya).score(k.transform(Xb), yb)

```

</details>

### Exercise 3 · Stretch · Tune gamma

For `gamma` in [0.01, 0.1, 1, 10, 100] compute the test accuracy of the Kernel-PCA + logistic pipeline. Store the results in `acc_by_gamma` (dict) and the best gamma in `best_gamma`.

In [ ]:
acc_by_gamma = {}
best_gamma = None   # TODO


In [ ]:
try:
    check("five values", len(acc_by_gamma) == 5)
    check("tiny gamma is close to linear PCA", acc_by_gamma[0.01] < 0.8)
    check("best gamma is good", acc_by_gamma[best_gamma] > 0.95)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
acc_by_gamma = {}
for g in (0.01, 0.1, 1, 10, 100):
    k = KernelPCA(n_components=2, kernel="rbf", gamma=g).fit(Xa)
    acc_by_gamma[g] = LogisticRegression().fit(k.transform(Xa), ya).score(k.transform(Xb), yb)
best_gamma = max(acc_by_gamma, key=acc_by_gamma.get)

```

Gamma sets how local the similarity is: too small behaves like a linear kernel, too large memorises individual points.

</details>

---
*Back to the course: **Machine Learning End To End → Kernel PCA**.*